# Non-pregnant anemia prevalence and DALYs averted by fortification

We take a "multiplication model" approach here, shifting continuous hemoglobin (as estimated
by GBD) and seeing what impact that has on anemia.

It's important to note that we directly use hemoglobin estimates, which are the first step
of the GBD anemia estimation pipeline. The risks and causes that are related to anemia are
all calculated downstream from this.

In [ ]:
import gbd_mapping
import risk_distributions
import pathlib
import pandas as pd, numpy as np
import vivarium_inputs
from vivarium_inputs import utility_data, globals as vi_globals, utilities as vi_utils
from vivarium_gbd_access import gbd
import os, contextlib, warnings, loguru
from lsff_utils import config_utils
from lsff_utils.hemoglobin_distribution import hemoglobin_cdf_from_mean_sd

from vivarium_inputs.validation.raw import DataDoesNotExistError, DataAbnormalError
from tqdm.notebook import tqdm

In [ ]:
pd.set_option("display.max_columns", 30)

In [ ]:
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

In [ ]:
location = "ethiopia"
vehicle = "salt"

In [ ]:
intervention_scenarios = config_utils.get_config()["custom_intervention_scenarios"].get(
    location, ["intervention"]
)
intervention_scenarios

## Setup and scenarios

In [ ]:
DRAWS = [
    f"draw_{i}" for i in range(500)
]  # NOTE: Some GBD 2021 things return 1,000 but others don't

In [ ]:
effective_coverage_baseline = pd.read_csv(
    f"../0100_data_prep/results/folate/{vehicle}/baseline_fortification/effective_coverage/{location}.csv"
)
assert (effective_coverage_baseline.vehicle_name == vehicle).all()
effective_coverage_baseline = effective_coverage_baseline.drop(columns=["vehicle_name"])
effective_coverage_baseline

In [ ]:
def expand(df):
    for col in sorted(list(set(df.columns) - {"value"})):
        if df[col].isnull().any():
            df = pd.concat(
                [
                    df[df[col].notnull()],
                    *[
                        df[df[col].isnull()].assign(**{col: value})
                        for value in df[df[col].notnull()][col].unique()
                    ],
                ]
            )

    return df

In [ ]:
for col, fill_value in [("age_start", 0), ("age_end", 125)]:
    if col not in effective_coverage_baseline.columns:
        effective_coverage_baseline[col] = fill_value
    else:
        effective_coverage_baseline[col] = effective_coverage_baseline[col].fillna(
            fill_value
        )

In [ ]:
effective_coverage_baseline = expand(effective_coverage_baseline)
effective_coverage_baseline

In [ ]:
effective_coverage_intervention = pd.concat(
    [
        pd.read_csv(
            f"../0100_data_prep/results/folate/{vehicle}/{intervention_scenario}/intervention_fortification/effective_coverage/{location}.csv"
        ).assign(scenario=intervention_scenario)
        for intervention_scenario in intervention_scenarios
    ]
)
assert (effective_coverage_intervention.vehicle_name == vehicle).all()
effective_coverage_intervention = effective_coverage_intervention.drop(
    columns=["vehicle_name"]
)
effective_coverage_intervention

In [ ]:
for col, fill_value in [("age_start", 0), ("age_end", 125)]:
    if col not in effective_coverage_intervention.columns:
        effective_coverage_intervention[col] = fill_value
    else:
        effective_coverage_intervention[col] = effective_coverage_intervention[
            col
        ].fillna(fill_value)

In [ ]:
effective_coverage_intervention = expand(effective_coverage_intervention)
effective_coverage_intervention

In [ ]:
population = pd.read_csv(
    f"../0100_data_prep/results/population/stratified/{location}.csv"
)

In [ ]:
non_pregnant_pop = population.pipe(lambda df: df[df.pregnant == "not_pregnant"]).drop(
    columns="pregnant"
)
non_pregnant_pop = non_pregnant_pop.set_index(
    [c for c in non_pregnant_pop.columns if c != "value"]
).value
non_pregnant_pop

In [ ]:
population.set_index(["sex", "age_start", "age_end"]).loc[
    ("Female", 25, 30)
].value.sum()

In [ ]:
non_pregnant_pop.loc[("Female", 25, 30)][1]

In [ ]:
non_pregnant_pop.loc[("Female", 25, 30)].sum()

In [ ]:
population_age_groups = (
    non_pregnant_pop.reset_index()[["age_start", "age_end"]]
    .drop_duplicates()
    .sort_values("age_start")
)
population_age_groups

In [ ]:
def map_to_population_age_groups(df):
    result = (
        population_age_groups.merge(df, how="cross", suffixes=("", "_orig"))
        .pipe(
            lambda df: df[
                (df.age_end <= df.age_end_orig) & (df.age_start >= df.age_start_orig)
            ]
        )
        .drop(columns=["age_start_orig", "age_end_orig"])
    )
    return result

In [ ]:
effective_coverage_baseline = (
    map_to_population_age_groups(effective_coverage_baseline)
    .set_index([c for c in effective_coverage_baseline.columns if c != "value"])
    .value
)
effective_coverage_intervention = (
    map_to_population_age_groups(effective_coverage_intervention)
    .set_index([c for c in effective_coverage_intervention.columns if c != "value"])
    .value
)

In [ ]:
if "sex" not in effective_coverage_baseline.index.names:
    # Assume does not vary
    effective_coverage_baseline = pd.concat(
        [
            effective_coverage_baseline.reset_index()
            .assign(sex="Female")
            .set_index(["wealth_quintile", "sex", "age_start", "age_end"])
            .value,
            effective_coverage_baseline.reset_index()
            .assign(sex="Male")
            .set_index(["wealth_quintile", "sex", "age_start", "age_end"])
            .value,
        ]
    )

In [ ]:
effective_coverage_baseline = (
    effective_coverage_baseline.reset_index()
    .set_index(["sex", "age_start", "age_end", "wealth_quintile"])
    .value
)

In [ ]:
if "sex" not in effective_coverage_intervention.index.names:
    # Assume does not vary
    effective_coverage_intervention = pd.concat(
        [
            effective_coverage_intervention.reset_index()
            .assign(sex="Female")
            .set_index(["sex", "age_start", "age_end", "wealth_quintile"])
            .value,
            effective_coverage_intervention.reset_index()
            .assign(sex="Male")
            .set_index(["sex", "age_start", "age_end", "wealth_quintile"])
            .value,
        ]
    )

In [ ]:
effective_coverage_intervention = (
    effective_coverage_intervention.reset_index()
    # TODO: Make this clearer. This is just reordering.
    .set_index(["sex", "age_start", "age_end", "wealth_quintile", "scenario"]).value
)

In [ ]:
example_sex = "Female"
example_age_start = 25
example_age_end = 30
example_wealth_quintile = 1
example_scenario = effective_coverage_intervention.index.get_level_values("scenario")[0]
example_tuple = (
    example_sex,
    example_age_start,
    example_age_end,
    example_wealth_quintile,
)

In [ ]:
effective_coverage_baseline.loc[example_tuple]

In [ ]:
effective_coverage_intervention.loc[example_tuple]

In [ ]:
def reshape_to_vivarium_format(df, location):
    df = vi_utils.reshape(df, value_cols=[c for c in df.columns if "draw_" in c])
    df = vi_utils.scrub_gbd_conventions(df, location)
    df = vi_utils.split_interval(df, interval_column="age", split_column_prefix="age")
    df = vi_utils.split_interval(df, interval_column="year", split_column_prefix="year")
    df = vi_utils.sort_hierarchical_data(df)
    df.index = df.index.droplevel("location")
    return df

## Pull GBD hemoglobin distributions

In [ ]:
me_ids = {
    "hemoglobin_mean": 10487,
    "hemoglobin_sd": 10488,
}

In [ ]:
def get_modelable_entity_draws(me_id, location):
    location_id = utility_data.get_location_id(location.title())
    result = gbd.get_modelable_entity_draws(
        me_id=me_id, location_id=location_id, year_id=2021
    )
    return (
        reshape_to_vivarium_format(result, location.title())
        .droplevel(
            [
                "year_start",
                "year_end",
                "measure_id",
                "metric_id",
                "model_version_id",
                "modelable_entity_id",
            ]
        )[DRAWS]
        .copy()
    )

In [ ]:
hgb_mean = get_modelable_entity_draws(me_ids["hemoglobin_mean"], location)
hgb_mean

In [ ]:
hgb_mean.loc[("Female", 25, 30)].mean()

In [ ]:
hemoglobin_mean_disparities = pd.read_csv(
    f"../0100_data_prep/results/hemoglobin/mean_disparities/{location}.csv"
)
hemoglobin_mean_disparities = (
    map_to_population_age_groups(
        hemoglobin_mean_disparities[
            hemoglobin_mean_disparities.pregnant == "not_pregnant"
        ].drop(columns=["pregnant"])
    )
    .set_index(["sex", "age_start", "age_end", "wealth_quintile"])
    .value
)
hemoglobin_mean_disparities

In [ ]:
wealth_quintile_probabilities = pd.read_csv(
    f"../0100_data_prep/results/wealth_quintile_probabilities/{location}.csv"
)
wealth_quintile_probabilities

In [ ]:
wealth_quintile_probabilities = map_to_population_age_groups(
    wealth_quintile_probabilities[
        wealth_quintile_probabilities.pregnant == "not_pregnant"
    ].drop(columns=["pregnant"])
).set_index(["sex", "age_start", "age_end"])
wealth_quintile_probabilities.columns.name = "wealth_quintile"
wealth_quintile_probabilities = wealth_quintile_probabilities.stack()
# TODO: Store wealth quintile probabilities without using numbers as columns
wealth_quintile_probabilities = (
    wealth_quintile_probabilities.rename("value")
    .reset_index()
    .assign(wealth_quintile=lambda df: df.wealth_quintile.astype(int))
    .set_index(wealth_quintile_probabilities.index.names)
    .value
)
wealth_quintile_probabilities

In [ ]:
assert np.allclose(
    wealth_quintile_probabilities.groupby(["sex", "age_start", "age_end"]).sum(), 1.0
)

In [ ]:
def distribute_by_disparities(df, disparities):
    pre_disparity_groups = (
        df.mul(wealth_quintile_probabilities, axis=0)
        .groupby([c for c in df.index.names if c != "wealth_quintile"])
        .sum()
    )
    print("Before distributing by disparities:")
    display(pre_disparity_groups)

    df = df.mul(disparities, axis=0)

    scale_factor = (
        pre_disparity_groups
        / df.mul(wealth_quintile_probabilities, axis=0)
        .groupby([c for c in df.index.names if c != "wealth_quintile"])
        .sum()
    )
    print(f"Scale factor: {scale_factor}")

    df = df * scale_factor

    assert np.allclose(
        df.mul(wealth_quintile_probabilities, axis=0)
        .groupby([c for c in df.index.names if c != "wealth_quintile"])
        .sum(),
        pre_disparity_groups,
    )

    return df

In [ ]:
hgb_mean = distribute_by_disparities(hgb_mean, hemoglobin_mean_disparities)

In [ ]:
hgb_mean.columns.name = "draw"
hgb_mean = hgb_mean.stack().rename("mean")

In [ ]:
hgb_sd = get_modelable_entity_draws(me_ids["hemoglobin_sd"], location)
hgb_sd

In [ ]:
hemoglobin_sd_disparities = pd.read_csv(
    f"../0100_data_prep/results/hemoglobin/sd_disparities/{location}.csv"
)
hemoglobin_sd_disparities = (
    map_to_population_age_groups(
        hemoglobin_sd_disparities[
            hemoglobin_sd_disparities.pregnant == "not_pregnant"
        ].drop(columns=["pregnant"])
    )
    .set_index(["sex", "age_start", "age_end", "wealth_quintile"])
    .value
)
hemoglobin_sd_disparities

In [ ]:
hgb_sd = distribute_by_disparities(hgb_sd, hemoglobin_sd_disparities)
hgb_sd

In [ ]:
hgb_sd.columns.name = "draw"
hgb_sd = hgb_sd.stack().rename("sd")
hgb_sd

In [ ]:
hgb_mean.loc[("Female", 25, 30, 1)].mean()

In [ ]:
hgb_sd.loc[("Female", 25, 30, 1)].mean()

## Effect size

**This is extremely speculative and has major gaps.**
In particular, we don't model any kind of individual heterogeneity
(folate-responsiveness) which is almost certainly at play, instead
increasing everyone's hemoglobin by the same amount.

Plus, [a study in Ethiopia](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8990104/#sup1)
that found a significant association between serum folate and anemia also found *no*
association between dietary folate intake and anemia, which raises questions about
whether fortification would help.

[The study we rely on here](https://www.cambridge.org/core/journals/british-journal-of-nutrition/article/micronutrients-and-sociodemographic-factors-were-major-predictors-of-anaemia-among-the-ethiopian-population/835E8A87F9C99213A5C5CD34DF2520CC#article)
only measured serum folate in non-pregnant WRA.
We extrapolate this to men in the same age group, based on the first study linked,
which found a similar amount of serum-folate-attributable anemia in that population.
However, we do not apply it outside of this age group, because it was found not to be
influential in children.

In [ ]:
assert location == "ethiopia"

In [ ]:
# Table 6 of https://cdn.nutrition.org/article/S2475-2991%2824%2901728-1/fulltext
s_baseline_folate_intake = pd.Series(
    {
        1: 166,
        2: 152,
        3: 137,
        4: 350,
        5: 469,  # NRV is 400 mcg/day
    }
)
s_baseline_folate_intake.index.name = "wealth_quintile"

In [ ]:
effective_coverage_baseline

In [ ]:
eff_fort_baseline_path = f"../0100_data_prep/results/folate/{vehicle}/baseline_fortification/effective_coverage/{location}.csv"

df_eff_fort_baseline = pd.read_csv(eff_fort_baseline_path)
assert (df_eff_fort_baseline.vehicle_name == vehicle).all()
df_eff_fort_baseline = df_eff_fort_baseline.drop(columns=["vehicle_name"])
df_eff_fort_baseline

In [ ]:
df_eff_fort_intervention = pd.concat(
    [
        pd.read_csv(
            f"../0100_data_prep/results/folate/{vehicle}/{intervention_scenario}/intervention_fortification/effective_coverage/{location}.csv"
        ).assign(scenario=intervention_scenario)
        for intervention_scenario in intervention_scenarios
    ]
)
assert (df_eff_fort_intervention.vehicle_name == vehicle).all()
df_eff_fort_intervention = df_eff_fort_intervention.drop(columns=["vehicle_name"])
df_eff_fort_intervention

In [ ]:
# Based on above evidence that found no effect in kids, we limit ourselves to 15-50
if "age_start" in df_eff_fort_baseline.columns:
    df_eff_fort_baseline = df_eff_fort_baseline[
        (df_eff_fort_baseline.age_start >= 15) & (df_eff_fort_baseline.age_end <= 50)
    ]

if "age_start" in df_eff_fort_intervention.columns:
    df_eff_fort_intervention = df_eff_fort_intervention[
        (df_eff_fort_intervention.age_start >= 15)
        & (df_eff_fort_intervention.age_end <= 50)
    ]

In [ ]:
df_eff_fort_baseline = df_eff_fort_baseline.set_index(
    [c for c in df_eff_fort_baseline.columns if c != "value"]
).value
df_eff_fort_intervention = df_eff_fort_intervention.set_index(
    [c for c in df_eff_fort_intervention.columns if c != "value"]
).value

In [ ]:
s_daily_vehicle = pd.read_csv(
    f"../0100_data_prep/results/{vehicle}/vehicle_consumption/amount/mean/{location}.csv"
)
assert (s_daily_vehicle.vehicle_name == vehicle).all()
s_daily_vehicle = s_daily_vehicle.drop(columns=["vehicle_name"])
s_daily_vehicle = s_daily_vehicle.set_index(
    [c for c in s_daily_vehicle.columns if c != "value"]
).value
s_daily_vehicle

In [ ]:
# NOTE: We assume these values (for WRA) apply to all ages and sexes!
s_daily_vehicle = s_daily_vehicle.droplevel(["sex", "age_start", "age_end"])
s_daily_vehicle

In [ ]:
# NOTE: We do not back-calculate this from NTD rates, which means this may not be consistent
# with our RBC folate assumptions!
# However, we did a quick comparison of reported RBC folate with our back-calculated from NTD
# rates and it seemed pretty consistent.
serum_folate_baseline_tertiles = [
    12.3,
    11.5,
    12.6,
]  # Table 1 of https://www.sciencedirect.com/science/article/pii/S2475299122000610#t1; note this is in WRA and is a median!
serum_folate_baseline = pd.Series(
    np.interp(
        [0.1, 0.3, 0.5, 0.7, 0.9],
        [0.165, 0.33 + 0.165, 0.66 + 0.165],
        serum_folate_baseline_tertiles,
    ),  # interpolate without extrapolating
    index=[1, 2, 3, 4, 5],
)
serum_folate_baseline.index.name = "wealth_quintile"
serum_folate_baseline

In [ ]:
# Fortification folate needs to be converted into dietary folate equivalents (DFEs)
# for use with our effect size.
# https://www.jandonline.org/article/S0002-8223(00)00027-4/pdf
fortification_mcg_to_dfe = 1.7

In [ ]:
baseline_concentration_mcg_per_gram = pd.read_csv(
    f"../0100_data_prep/results/folate/{vehicle}/baseline_fortification/concentration/{location}.csv"
)
assert (baseline_concentration_mcg_per_gram.vehicle_name == vehicle).all()
assert baseline_concentration_mcg_per_gram.value.nunique() == 1
baseline_concentration_mcg_per_gram = baseline_concentration_mcg_per_gram.value.iloc[0]
baseline_concentration_mcg_per_gram

In [ ]:
intervention_concentration_mcg_per_gram = pd.concat(
    [
        pd.read_csv(
            f"../0100_data_prep/results/folate/{vehicle}/{intervention_scenario}/intervention_fortification/concentration/{location}.csv"
        ).assign(scenario=intervention_scenario)
        for intervention_scenario in intervention_scenarios
    ]
)
assert (intervention_concentration_mcg_per_gram.vehicle_name == vehicle).all()
intervention_concentration_mcg_per_gram = (
    intervention_concentration_mcg_per_gram.drop(columns=["vehicle_name"])
    .set_index("scenario")
    .value
)
intervention_concentration_mcg_per_gram

In [ ]:
df_eff_fort_baseline

In [ ]:
effective_coverage_baseline

In [ ]:
s_baseline_folate_intake

In [ ]:
# NOTE: We delete intake here that we believe to be baked into
# our baseline values. Since this has only been run for Ethiopia-salt,
# this is a no-op (no baseline fortification with folate).
# In the future we might revisit this to ensure what we are deleting
# is appropriate to our data source.
s_zero_folate_intake = s_baseline_folate_intake - (
    df_eff_fort_baseline
    * s_daily_vehicle
    * baseline_concentration_mcg_per_gram
    * fortification_mcg_to_dfe
)
s_zero_folate_intake

In [ ]:
s_intervention_folate_intake = s_zero_folate_intake + (
    df_eff_fort_intervention
    * intervention_concentration_mcg_per_gram
    * fortification_mcg_to_dfe
    * s_daily_vehicle
)
s_intervention_folate_intake

In [ ]:
zero_folate_intake_pct_decrease = (
    s_baseline_folate_intake - s_zero_folate_intake
) / s_baseline_folate_intake

In [ ]:
intevention_folate_intake_pct_increase_from_zero = (
    s_intervention_folate_intake - s_zero_folate_intake
) / s_baseline_folate_intake
intevention_folate_intake_pct_increase_from_zero

In [ ]:
# https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4425166/
# These results indicate that for every 10% increase in natural food folate intake, serum/plasma folate concentrations could increase by approximately 7%
serum_folate_zero = serum_folate_baseline / (
    1 + ((7 / 10) * zero_folate_intake_pct_decrease)
)
serum_folate_zero

In [ ]:
serum_folate_intervention = serum_folate_zero * (
    1 + ((7 / 10) * intevention_folate_intake_pct_increase_from_zero)
)
serum_folate_intervention

In [ ]:
# Table 5, https://www.cambridge.org/core/journals/british-journal-of-nutrition/article/micronutrients-and-sociodemographic-factors-were-major-predictors-of-anaemia-among-the-ethiopian-population/835E8A87F9C99213A5C5CD34DF2520CC#article
# In WRA!
# NOTE: I think they do not explicitly state their serum folate unit,
# but they describe deficiency in terms of nmol/l
# Hemoglobin in g/dL in the paper, so we multiply by 10 to get g/L
hemoglobin_shift_g_l_per_serum_folate_nmol_l = 0.02 * 10

In [ ]:
if location == "india" and vehicle == "rice":
    # Confusingly, our baseline scenario (our best guess about the present)
    # is *not* a good guess about 2021 (the year of our GBD hemoglobin estimate),
    # because this program has rolled out almost entirely since then:
    # In the phase-I of the roll out, the fortified rice was introduced in the social welfare schemes such as Integrated Child Development Scheme (ICDS)
    # and Pradhan Mantri Poshan Shakti Nirman (PM POSHAN, earlier known as the National Program of Mid-Day Meal in Schools)
    # throughout India during 2021–22 [18].
    # Phase-II has covered aspirational and high burden districts for anemia (total 291 districts) under Public Distribution System (PDS) and other welfare schemes,
    # in addition to Phase-I districts, by March 2023 [18].
    # All the remaining districts in India will be covered in Phase III by March 2024 [19].
    # ~ https://pmc.ncbi.nlm.nih.gov/articles/PMC11305529/
    effective_coverage_baseline_2021 = 0
else:
    effective_coverage_baseline_2021 = effective_coverage_baseline

In [ ]:
baseline_2021_hemoglobin_contribution = (
    effective_coverage_baseline_2021
    * (serum_folate_baseline - serum_folate_zero)
    * hemoglobin_shift_g_l_per_serum_folate_nmol_l
)
baseline_2021_hemoglobin_contribution[
    (baseline_2021_hemoglobin_contribution.index.get_level_values("age_end") <= 15)
    | (baseline_2021_hemoglobin_contribution.index.get_level_values("age_start") >= 50)
] = 0
baseline_2021_hemoglobin_contribution

In [ ]:
# Delete the fortification effects on hemoglobin baked into the GBD 2021
# hemoglobin estimate
hgb_mean_zero_fort = (hgb_mean - baseline_2021_hemoglobin_contribution).rename("mean")
hgb_mean_zero_fort

In [ ]:
intervention_hemoglobin_contribution_vs_zero = (
    effective_coverage_intervention
    * (serum_folate_intervention - serum_folate_zero)
    * hemoglobin_shift_g_l_per_serum_folate_nmol_l
)
intervention_hemoglobin_contribution_vs_zero[
    (
        intervention_hemoglobin_contribution_vs_zero.index.get_level_values("age_end")
        <= 15
    )
    | (
        intervention_hemoglobin_contribution_vs_zero.index.get_level_values("age_start")
        >= 50
    )
] = 0
intervention_hemoglobin_contribution_vs_zero

In [ ]:
intervention_hemoglobin_contribution_vs_zero.sort_values()

In [ ]:
intervention_hemoglobin_contribution_vs_zero[
    (intervention_hemoglobin_contribution_vs_zero > 0)
    & (
        intervention_hemoglobin_contribution_vs_zero.index.get_level_values("scenario")
        == "intervention_25_nrv"
    )
].describe()

In [ ]:
intervention_hemoglobin_contribution_vs_zero[
    (intervention_hemoglobin_contribution_vs_zero > 0)
    & (
        intervention_hemoglobin_contribution_vs_zero.index.get_level_values("scenario")
        == "intervention_100_nrv"
    )
].describe()

In [ ]:
hgb_mean_with_fort = hgb_mean_zero_fort.add(
    intervention_hemoglobin_contribution_vs_zero
).rename("mean")
hgb_mean_with_fort

In [ ]:
thresholds = (
    reshape_to_vivarium_format(
        pd.read_csv("/share/mnch/anemia/code/reference/model/anemia_thresholds.csv"),
        location.title(),
    )
    .droplevel(["age_group_name", "grp"])
    .reset_index()
)
thresholds

In [ ]:
assert (thresholds.hgb_upper_mild == thresholds.hgb_upper_anemic).all() & (
    thresholds.hgb_lower_severe == thresholds.hgb_lower_anemic
).all()
thresholds = thresholds.drop(columns=["hgb_upper_anemic", "hgb_lower_anemic"])

In [ ]:
assert (thresholds.hgb_lower_mild == thresholds.hgb_upper_moderate).all() & (
    thresholds.hgb_lower_moderate == thresholds.hgb_upper_severe
).all()
thresholds = thresholds.drop(columns=["hgb_lower_mild", "hgb_lower_moderate"])

In [ ]:
thresholds = thresholds.set_index(["sex", "age_start", "age_end", "pregnant"])
thresholds

In [ ]:
def calculate_anemia_from_mean_sd_hemoglobin(mean, sd):
    orig_index = mean.index
    result = (
        mean.reset_index()
        .merge(sd.reset_index(), how="outer", validate="m:1")
        .assign(pregnant=0)
        .merge(thresholds.reset_index(), how="left", validate="m:1")
    )

    cdf = hemoglobin_cdf_from_mean_sd(result["mean"], result.sd)

    result["severe"] = cdf(result.hgb_upper_severe.copy()) - cdf(
        result.hgb_lower_severe.copy()
    )
    result["moderate"] = cdf(result.hgb_upper_moderate.copy()) - result["severe"].copy()
    result["mild"] = (
        cdf(result.hgb_upper_mild.copy())
        - result["moderate"].copy()
        - result["severe"].copy()
    )
    result["anemic"] = result["mild"] + result["moderate"] + result["severe"]

    return result.set_index(orig_index.names)[["severe", "moderate", "mild", "anemic"]]

In [ ]:
hgb_sd.hist()

In [ ]:
np.percentile(hgb_sd, 1)

In [ ]:
# Some SDs are super super small! They make anemia not computable
hgb_sd = hgb_sd.clip(lower=np.percentile(hgb_sd, 1))

In [ ]:
zero_fort_anemia = calculate_anemia_from_mean_sd_hemoglobin(
    hgb_mean_zero_fort.rename("mean"), hgb_sd
)
zero_fort_anemia

In [ ]:
with_fort_anemia = calculate_anemia_from_mean_sd_hemoglobin(
    hgb_mean_with_fort.rename("mean"), hgb_sd
)
with_fort_anemia

In [ ]:
total_population_anemia = calculate_anemia_from_mean_sd_hemoglobin(hgb_mean, hgb_sd)
total_population_anemia

In [ ]:
# NOTE: Here, we assume everyone is folate-responsive, instead of splitting this effect into
# a bigger effect among the folate-responsive and 0 among the non-

In [ ]:
baseline_anemia = zero_fort_anemia
# TODO: If there were baseline coverage, we'd need to know
# its concentration!
# (
#     zero_fort_anemia.mul(
#         (1 - effective_coverage_baseline), axis=0
#     )
#     + with_fort_anemia.mul(
#         effective_coverage_baseline, axis=0
#     )
# )
baseline_anemia

In [ ]:
(baseline_anemia - total_population_anemia).describe()

In [ ]:
(
    baseline_anemia.loc[example_tuple].mean()
    - with_fort_anemia.loc[example_tuple].mean()
) / baseline_anemia.loc[example_tuple].mean()

In [ ]:
zero_fort_anemia

In [ ]:
intervention_anemia = zero_fort_anemia.mul(
    (1 - effective_coverage_intervention),
    axis=0,
) + with_fort_anemia.mul(effective_coverage_intervention, axis=0)
intervention_anemia

In [ ]:
from lsff_utils.hemoglobin_distribution import hemoglobin_pdf_from_mean_sd

In [ ]:
total_population_pdf = hemoglobin_pdf_from_mean_sd(
    np.array([hgb_mean.loc[example_tuple].mean()]),
    np.array([hgb_sd.loc[example_tuple].mean()]),
)
total_population_pdf

In [ ]:
zero_fort_pdf = hemoglobin_pdf_from_mean_sd(
    np.array([hgb_mean_zero_fort.loc[example_tuple].mean()]),
    np.array([hgb_sd.loc[example_tuple].mean()]),
)
zero_fort_pdf

In [ ]:
with_fort_pdf = hemoglobin_pdf_from_mean_sd(
    np.array([hgb_mean_with_fort.loc[example_tuple].mean()]),
    np.array([hgb_sd.loc[example_tuple].mean()]),
)
with_fort_pdf

In [ ]:
def baseline_pdf(x):
    effective_covered = effective_coverage_baseline.loc[example_tuple]
    return (
        zero_fort_pdf(x) * (1 - effective_covered)
        + with_fort_pdf(x) * effective_covered
    )

In [ ]:
def intervention_pdf(x):
    effective_covered = effective_coverage_intervention.loc[example_tuple]
    return (
        zero_fort_pdf(x) * (1 - effective_covered)
        + with_fort_pdf(x) * effective_covered
    )

In [ ]:
import matplotlib.pyplot as plt

x_values = np.linspace(60, 160, 100)
with np.errstate(under="ignore"):
    plt.plot(
        x_values,
        [total_population_pdf(x) for x in x_values],
        label="Total population from GBD",
    )
    plt.plot(
        x_values,
        [zero_fort_pdf(x) for x in x_values],
        label="No fortification",
    )
    plt.plot(
        x_values,
        [with_fort_pdf(x) for x in x_values],
        label="With fortification",
    )
    plt.plot(
        x_values,
        [baseline_pdf(x) for x in x_values],
        label="Hemoglobin in baseline scenario",
    )
    plt.plot(
        x_values,
        [intervention_pdf(x) for x in x_values],
        label="Hemoglobin in intervention scenario",
    )
    plt.vlines(
        thresholds.loc[
            (example_sex, example_age_start, example_age_end, 0)
        ].hgb_upper_mild,
        0,
        0.03,
        linestyles="dashed",
        label="Anemia",
        color="lime",
    )
    plt.vlines(
        thresholds.loc[
            (example_sex, example_age_start, example_age_end, 0)
        ].hgb_upper_moderate,
        0,
        0.03,
        linestyles="dashed",
        label="Moderate anemia",
        color="gold",
    )
    plt.vlines(
        thresholds.loc[
            (example_sex, example_age_start, example_age_end, 0)
        ].hgb_upper_severe,
        0,
        0.03,
        linestyles="dashed",
        label="Severe anemia",
        color="crimson",
    )
    plt.xlabel("Hemoglobin (g/L)")
    plt.ylabel("Probability density")
    plt.title(
        f"Hemoglobin before and after {vehicle} fortification intervention in non-pregnant females 25-30 years in the bottom quintile, {location.title()}"
    )
    plt.legend(bbox_to_anchor=(1.05, 1))

In [ ]:
(baseline_anemia - intervention_anemia).sort_values("anemic")

In [ ]:
def anemia_to_yld_rates(anemia):
    disability_weights = pd.read_hdf(
        "/mnt/team/simulation_science/costeffectiveness/auxiliary_data/GBD_2021/02_processed_data/disability_weight/sequela/all/all.hdf"
    )
    disability_weights = (
        disability_weights[
            disability_weights.healthstate.isin(
                ["anemia_mild", "anemia_mod", "anemia_sev"]
            )
        ]
        .set_index("healthstate")
        .filter(like="draw_")
    )
    disability_weights.columns.name = "draw"
    disability_weights = (
        disability_weights.stack().rename("disability_weight").reset_index()
    )
    display(disability_weights)

    orig_index = anemia.index
    anemia = (
        anemia.reset_index()
        .merge(
            disability_weights[disability_weights.healthstate == "anemia_mild"][
                ["draw", "disability_weight"]
            ].rename(columns={"disability_weight": "mild_dw"}),
            validate="m:1",
        )
        .merge(
            disability_weights[disability_weights.healthstate == "anemia_mod"][
                ["draw", "disability_weight"]
            ].rename(columns={"disability_weight": "moderate_dw"}),
            validate="m:1",
        )
        .merge(
            disability_weights[disability_weights.healthstate == "anemia_sev"][
                ["draw", "disability_weight"]
            ].rename(columns={"disability_weight": "severe_dw"}),
            validate="m:1",
        )
    )

    anemia["mild_yld_rate"] = anemia.mild * anemia.mild_dw
    anemia["moderate_yld_rate"] = anemia.moderate * anemia.moderate_dw
    anemia["severe_yld_rate"] = anemia.severe * anemia.severe_dw
    anemia["anemic_yld_rate"] = (
        anemia["mild_yld_rate"]
        + anemia["moderate_yld_rate"]
        + anemia["severe_yld_rate"]
    )

    return anemia.set_index(orig_index.names).filter(like="yld_rate")

In [ ]:
zero_fort_anemia_yld_rates = anemia_to_yld_rates(zero_fort_anemia)
zero_fort_anemia_yld_rates

In [ ]:
baseline_anemia_yld_rates = anemia_to_yld_rates(baseline_anemia)
baseline_anemia_yld_rates

In [ ]:
(
    baseline_anemia_yld_rates.loc[("Female", 25, 30, 1)].mean()
    - anemia_to_yld_rates(with_fort_anemia).loc[("Female", 25, 30, 1)].mean()
) / baseline_anemia_yld_rates.loc[("Female", 25, 30, 1)].mean()

In [ ]:
intervention_anemia_yld_rates = anemia_to_yld_rates(intervention_anemia)
intervention_anemia_yld_rates

In [ ]:
(baseline_anemia_yld_rates - intervention_anemia_yld_rates).sort_values(
    "anemic_yld_rate"
)

In [ ]:
(baseline_anemia_yld_rates - intervention_anemia_yld_rates).anemic_yld_rate.describe()

In [ ]:
assert (
    ((baseline_anemia_yld_rates - intervention_anemia_yld_rates).anemic_yld_rate >= 0)
    | np.isclose(
        (baseline_anemia_yld_rates - intervention_anemia_yld_rates).anemic_yld_rate, 0
    )
).all()

In [ ]:
zero_fort_ylds = (
    zero_fort_anemia_yld_rates.anemic_yld_rate.unstack("draw").mean(axis=1)
    * non_pregnant_pop
)
zero_fort_ylds

In [ ]:
baseline_ylds = (
    baseline_anemia_yld_rates.anemic_yld_rate.unstack("draw").mean(axis=1)
    * non_pregnant_pop
)
baseline_ylds

In [ ]:
intervention_ylds = (
    intervention_anemia_yld_rates.anemic_yld_rate.unstack("draw").mean(axis=1)
    * non_pregnant_pop
)
intervention_ylds

In [ ]:
(
    baseline_ylds.loc[("Female", 25, 30, 1)]
    - intervention_ylds.loc[("Female", 25, 30, 1)]
)

In [ ]:
(
    baseline_ylds.groupby(["wealth_quintile"]).sum()
    - intervention_ylds.groupby(["wealth_quintile", "scenario"]).sum()
)

In [ ]:
ylds = pd.concat(
    [
        zero_fort_ylds.rename("value").reset_index().assign(scenario="zero"),
        baseline_ylds.rename("value").reset_index().assign(scenario="baseline"),
        intervention_ylds.rename("value").reset_index(),
    ],
    ignore_index=True,
)
ylds

In [ ]:
results_dir = f"./results/{vehicle.lower()}/{location.lower()}"

In [ ]:
path = f"{results_dir}/ylds.parquet"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ylds.to_parquet(path)

In [ ]:
zero_fort_anemia_prevalence = zero_fort_anemia["anemic"].unstack("draw").mean(axis=1)
zero_fort_anemia_prevalence

In [ ]:
baseline_anemia_prevalence = baseline_anemia["anemic"].unstack("draw").mean(axis=1)
baseline_anemia_prevalence

In [ ]:
zero_fort_anemia_cases = zero_fort_anemia_prevalence.mul(non_pregnant_pop, axis=0)
zero_fort_anemia_cases

In [ ]:
zero_fort_anemia_cases.sum() / non_pregnant_pop.sum()

In [ ]:
baseline_anemia_cases = baseline_anemia_prevalence.mul(non_pregnant_pop, axis=0)
baseline_anemia_cases

In [ ]:
baseline_anemia_cases.sum() / non_pregnant_pop.sum()

In [ ]:
baseline_anemia_cases.groupby(["wealth_quintile"]).sum() / non_pregnant_pop.groupby(
    ["wealth_quintile"]
).sum()

In [ ]:
intervention_anemia_prevalence = (
    intervention_anemia["anemic"].unstack("draw").mean(axis=1)
)
intervention_anemia_prevalence

In [ ]:
total_population_anemia_prevalence = (
    total_population_anemia["anemic"].unstack("draw").mean(axis=1)
)
total_population_anemia_prevalence

In [ ]:
total_population_anemia_cases = total_population_anemia_prevalence.mul(
    non_pregnant_pop, axis=0
)
total_population_anemia_cases

In [ ]:
total_population_anemia_cases.sum() / non_pregnant_pop.sum()

In [ ]:
anemia_prevalence = pd.concat(
    [
        zero_fort_anemia_prevalence.rename("value")
        .reset_index()
        .assign(scenario="zero"),
        baseline_anemia_prevalence.rename("value")
        .reset_index()
        .assign(scenario="baseline"),
        intervention_anemia_prevalence.rename("value").reset_index(),
    ],
    ignore_index=True,
)
anemia_prevalence

In [ ]:
path = f"{results_dir}/anemia_prevalence.parquet"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
anemia_prevalence.to_parquet(path)

In [ ]:
intervention_anemia_cases = intervention_anemia_prevalence.mul(non_pregnant_pop, axis=0)
intervention_anemia_cases

In [ ]:
intervention_anemia_cases.groupby("scenario").sum() / non_pregnant_pop.sum()

In [ ]:
intervention_anemia_cases.groupby(
    ["scenario", "wealth_quintile"]
).sum() / non_pregnant_pop.groupby(["wealth_quintile"]).sum()

In [ ]:
(
    baseline_anemia_cases.groupby(["wealth_quintile"]).sum()
    - intervention_anemia_cases.groupby(["wealth_quintile"]).sum()
).map(lambda x: f"{round(x):,.0f}")

In [ ]:
anemia_cases = pd.concat(
    [
        zero_fort_anemia_cases.rename("value").reset_index().assign(scenario="zero"),
        baseline_anemia_cases.rename("value").reset_index().assign(scenario="baseline"),
        intervention_anemia_cases.rename("value").reset_index(),
    ],
    ignore_index=True,
)
anemia_cases

In [ ]:
path = f"{results_dir}/anemia_cases.parquet"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
anemia_cases.to_parquet(path)

In [ ]:
anemia_cases.notnull().all()